In [1]:
# Cell 1: Imports & Load
print("Cell 1: Imports & Load")
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import shapely
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from scipy.linalg import sqrtm
import joblib
from datetime import datetime

drive.mount("/content/drive")

# Paths
cleaned_path = '/content/drive/MyDrive/C3I-Internship-Work/Programs/MaritimeAnomalyDetection/data/cleaned-files/ais_cleaned_historical.csv'
features_dir = '/content/drive/MyDrive/C3I-Internship-Work/Programs/MaritimeAnomalyDetection/features'
os.makedirs(features_dir, exist_ok=True)

# Load TARGET (current Norwegian data)
df_target = pd.read_csv(cleaned_path, parse_dates=['poll_timestamp','msgtime'])
df_target = df_target.sort_values(['mmsi','poll_timestamp','msgtime'])
print(f"✅ Loaded TARGET (current) data: {df_target.shape[0]} rows, {df_target['mmsi'].nunique()} vessels")

# Load SOURCE (historical/original training data)
# NOTE: Adjust this path to your original training dataset
source_path = '/content/drive/MyDrive/C3I-Internship-Work/Programs/MaritimeAnomalyDetection/data/cleaned-files/ais_cleaned_source.csv'

# If no separate source file exists, use oldest portion of current data as source
if not os.path.exists(source_path):
    print("⚠️ No separate source file found. Using temporal split of current data.")
    df_target_sorted = df_target.sort_values('poll_timestamp')
    split_idx = len(df_target_sorted) // 2
    df_source = df_target_sorted.iloc[:split_idx].copy()
    df_target = df_target_sorted.iloc[split_idx:].copy()
    print(f"📊 Split data - Source: {len(df_source)} rows, Target: {len(df_target)} rows")
else:
    df_source = pd.read_csv(source_path, parse_dates=['poll_timestamp','msgtime'])
    df_source = df_source.sort_values(['mmsi','poll_timestamp','msgtime'])
    print(f"✅ Loaded SOURCE (historical) data: {df_source.shape[0]} rows, {df_source['mmsi'].nunique()} vessels")

df_target.head()

Cell 1: Imports & Load
Mounted at /content/drive
✅ Loaded TARGET (current) data: 1029 rows, 10 vessels
⚠️ No separate source file found. Using temporal split of current data.
📊 Split data - Source: 514 rows, Target: 515 rows


,courseOverGround,latitude,longitude,name,rateOfTurn,shipType,speedOverGround,trueHeading,navigationalStatus,mmsi,msgtime,poll_timestamp,poll_number,turn_rate
403,335.9,70.319242,22.875270,VALVAAGEN,NaN,90,0.0,NaN,1,257643700,2025-06-29 11:25:04+00:00,2025-06-29 11:25:09.591170+00:00,88,0.0
760,26.4,70.433720,21.109413,KV BISON,0.0,55,0.9,52.0,0,257934000,2025-06-29 11:25:02+00:00,2025-06-29 11:25:09.591170+00:00,88,-3.4
232,335.9,70.051945,20.633212,LIAVAAG,NaN,30,0.0,NaN,15,257567500,2025-06-29 11:25:04+00:00,2025-06-29 11:25:09.591170+00:00,88,0.0
78,44.8,70.417430,18.008653,NORDBAS,0.0,30,4.0,44.0,7,257032830,2025-06-29 11:25:08+00:00,2025-06-29 11:25:09.591170+00:00,88,0.7
79,47.1,70.418173,18.011077,NORDBAS,0.0,30,3.9,44.0,7,257032830,2025-06-29 11:26:08+00:00,2025-06-29 11:26:10.955748+00:00,89,2.3


In [2]:
# Cell 2: Feature Computations
print("Cell 2: Feature Computations for both SOURCE and TARGET datasets")

def compute_features(df, dataset_name):
    """Compute features for a given dataset"""
    print(f"Computing features for {dataset_name} dataset...")

    # 1) Delta lat/lon per hour
    df['delta_lat'] = df.groupby('mmsi')['latitude'].diff().fillna(0)
    df['delta_lon'] = df.groupby('mmsi')['longitude'].diff().fillna(0)
    df['time_diff'] = df.groupby('mmsi')['poll_timestamp'] \
                        .diff().dt.total_seconds().fillna(60) / 3600.0

    # 2) Movement vector (km/h approx)
    df['movement_vector'] = np.sqrt(df['delta_lat']**2 + df['delta_lon']**2) / df['time_diff']

    # 3) Speed over ground & turn_rate already present (rename for consistency)
    df.rename(columns={'speedOverGround':'sog','turn_rate':'turn_rate'}, inplace=True)

    # 4) EEZ violation & fishing_zone_dist placeholders
    df['eez_violation'] = 0   # all within Norwegian bounds
    df['fishing_zone_dist'] = 0   # compute if shapefiles available

    # 5) Keep only needed columns
    feature_cols = [
        'delta_lat','delta_lon','sog',
        'turn_rate','movement_vector',
        'eez_violation','fishing_zone_dist'
    ]
    df_feat = df[['mmsi','poll_timestamp'] + feature_cols].copy()

    # Remove any remaining NaNs or infinities
    df_feat = df_feat.replace([np.inf, -np.inf], np.nan)
    df_feat = df_feat.dropna()

    print(f"✅ {dataset_name} features computed: {df_feat.shape[0]} rows")
    return df_feat

# Compute features for both datasets
df_source_feat = compute_features(df_source, "SOURCE")
df_target_feat = compute_features(df_target, "TARGET")

# Define feature columns
feature_cols = [
    'delta_lat','delta_lon','sog',
    'turn_rate','movement_vector',
    'eez_violation','fishing_zone_dist'
]

print("✅ Feature computation complete for both datasets")
print(f"📊 Source features: {df_source_feat.shape}")
print(f"📊 Target features: {df_target_feat.shape}")

Cell 2: Feature Computations for both SOURCE and TARGET datasets
Computing features for SOURCE dataset...
✅ SOURCE features computed: 514 rows
Computing features for TARGET dataset...
✅ TARGET features computed: 515 rows
✅ Feature computation complete for both datasets
📊 Source features: (514, 9)
📊 Target features: (515, 9)


In [3]:
# ========================================================================
# CELL 2A: CORAL DOMAIN ADAPTATION
# Insert CORAL domain adaptation AFTER data cleaning but BEFORE scaling
# Aligns target (current) data distribution to source (historical) data
# ========================================================================

print("Cell 2A: CORAL Domain Adaptation")
print("="*60)

def apply_coral_adaptation(X_source, X_target, regularization=1e-6):
    """
    Apply CORAL (Correlation Alignment) domain adaptation

    Args:
        X_source: Source domain features (n_samples_s, n_features)
        X_target: Target domain features (n_samples_t, n_features)
        regularization: Small value added to diagonal for numerical stability

    Returns:
        X_target_coral: CORAL-adapted target features
        transform_info: Dictionary with transformation details
    """
    print(f"📊 Applying CORAL transformation...")
    print(f"   Source shape: {X_source.shape}")
    print(f"   Target shape: {X_target.shape}")

    # Step 1: Center both datasets
    mean_source = np.mean(X_source, axis=0)
    mean_target = np.mean(X_target, axis=0)

    X_source_centered = X_source - mean_source
    X_target_centered = X_target - mean_target

    # Step 2: Compute covariance matrices
    C_source = np.cov(X_source_centered, rowvar=False)
    C_target = np.cov(X_target_centered, rowvar=False)

    # Step 3: Add regularization for numerical stability
    C_source += regularization * np.eye(C_source.shape[0])
    C_target += regularization * np.eye(C_target.shape[0])

    print(f"   Covariance matrices computed with regularization: {regularization}")

    # Step 4: Compute matrix square roots
    try:
        C_source_sqrt = sqrtm(C_source).real
        C_target_inv_sqrt = sqrtm(np.linalg.inv(C_target)).real
    except Exception as e:
        print(f"❌ Matrix square root computation failed: {e}")
        print("   Trying eigenvalue decomposition...")

        # Alternative: Use eigenvalue decomposition
        eigenvals_s, eigenvecs_s = np.linalg.eigh(C_source)
        eigenvals_t, eigenvecs_t = np.linalg.eigh(C_target)

        # Ensure positive eigenvalues
        eigenvals_s = np.maximum(eigenvals_s, regularization)
        eigenvals_t = np.maximum(eigenvals_t, regularization)

        C_source_sqrt = eigenvecs_s @ np.diag(np.sqrt(eigenvals_s)) @ eigenvecs_s.T
        C_target_inv_sqrt = eigenvecs_t @ np.diag(1.0/np.sqrt(eigenvals_t)) @ eigenvecs_t.T

    # Step 5: Apply CORAL transformation
    # Whitening: X_target_white = X_target_centered @ C_target^(-1/2)
    X_target_white = X_target_centered @ C_target_inv_sqrt

    # Coloring: X_target_coral = X_target_white @ C_source^(1/2)
    X_target_coral_centered = X_target_white @ C_source_sqrt

    # Step 6: Add back source mean
    X_target_coral = X_target_coral_centered + mean_source

    # Compute alignment metrics
    C_coral = np.cov(X_target_coral - mean_source, rowvar=False)
    alignment_error = np.linalg.norm(C_coral - C_source, 'fro')

    transform_info = {
        'mean_source': mean_source,
        'mean_target': mean_target,
        'C_source': C_source,
        'C_target': C_target,
        'C_coral': C_coral,
        'alignment_error': alignment_error,
        'regularization': regularization
    }

    print(f"✅ CORAL transformation completed")
    print(f"   Frobenius alignment error: {alignment_error:.6f}")

    return X_target_coral, transform_info

# Apply CORAL adaptation
print(f"🔄 Starting CORAL domain adaptation...")
print(f"   Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Features: {feature_cols}")

# Extract feature matrices
X_source = df_source_feat[feature_cols].values
X_target = df_target_feat[feature_cols].values

print(f"📊 Data shapes before CORAL:")
print(f"   Source: {X_source.shape[0]} samples, {X_source.shape[1]} features")
print(f"   Target: {X_target.shape[0]} samples, {X_target.shape[1]} features")

# Apply CORAL transformation
X_target_coral, coral_info = apply_coral_adaptation(X_source, X_target)

# Replace target features with CORAL-adapted features
df_target_coral = df_target_feat.copy()
df_target_coral[feature_cols] = X_target_coral

print(f"✅ CORAL adaptation complete!")
print(f"   Original target covariance trace: {np.trace(coral_info['C_target']):.6f}")
print(f"   Source covariance trace: {np.trace(coral_info['C_source']):.6f}")
print(f"   CORAL-adapted covariance trace: {np.trace(coral_info['C_coral']):.6f}")

# Save CORAL transformation info
coral_info_path = os.path.join(features_dir, 'coral_transform_info.joblib')
joblib.dump(coral_info, coral_info_path)
print(f"💾 CORAL transformation info saved to: {coral_info_path}")

# Use CORAL-adapted features for downstream processing
df_feat = df_target_coral  # This replaces the original df_feat
print(f"📊 Using CORAL-adapted features for downstream processing: {df_feat.shape}")

Cell 2A: CORAL Domain Adaptation
🔄 Starting CORAL domain adaptation...
   Date: 2025-07-04 04:51:39
   Features: ['delta_lat', 'delta_lon', 'sog', 'turn_rate', 'movement_vector', 'eez_violation', 'fishing_zone_dist']
📊 Data shapes before CORAL:
   Source: 514 samples, 7 features
   Target: 515 samples, 7 features
📊 Applying CORAL transformation...
   Source shape: (514, 7)
   Target shape: (515, 7)
   Covariance matrices computed with regularization: 1e-06
✅ CORAL transformation completed
   Frobenius alignment error: 0.000005
✅ CORAL adaptation complete!
   Original target covariance trace: 3327.911197
   Source covariance trace: 4189.214969
   CORAL-adapted covariance trace: 4189.214960
💾 CORAL transformation info saved to: /content/drive/MyDrive/C3I-Internship-Work/Programs/MaritimeAnomalyDetection/features/coral_transform_info.joblib
📊 Using CORAL-adapted features for downstream processing: (515, 9)


In [4]:
# Cell 3: Normalize CORAL-Adapted Features & Save Norwegian Scaler
print("Cell 3: Normalize CORAL-Adapted Features & Save Norwegian Scaler")
print("="*60)

import os
import joblib
from sklearn.preprocessing import MinMaxScaler

# Define feature columns (same as before)
feature_cols = [
    'delta_lat', 'delta_lon', 'sog',
    'turn_rate', 'movement_vector',
    'eez_violation', 'fishing_zone_dist'
]

# Fit scaler on CORAL-adapted Norwegian data
scaler = MinMaxScaler()
scaler.fit(df_feat[feature_cols])

# Save scaler as a new file
features_dir = '/content/drive/MyDrive/C3I-Internship-Work/Programs/MaritimeAnomalyDetection/features'
os.makedirs(features_dir, exist_ok=True)
scaler_path = os.path.join(features_dir, 'norwegian_coral_scaler.joblib')
joblib.dump(scaler, scaler_path)

print(f"✅ Norwegian CORAL-adapted scaler saved to {scaler_path}")

# Apply scaling to CORAL-adapted feature columns
df_feat_scaled = df_feat.copy()
df_feat_scaled[feature_cols] = scaler.transform(df_feat[feature_cols])

print("✅ Scaled CORAL-adapted Norwegian features. Preview:")
print(df_feat_scaled[feature_cols].describe())

# Save CORAL-adapted features before scaling
coral_features_path = os.path.join(features_dir, 'norwegian_coral_features.csv')
df_feat.to_csv(coral_features_path, index=False)
print(f"💾 CORAL-adapted features saved to: {coral_features_path}")

Cell 3: Normalize CORAL-Adapted Features & Save Norwegian Scaler
✅ Norwegian CORAL-adapted scaler saved to /content/drive/MyDrive/C3I-Internship-Work/Programs/MaritimeAnomalyDetection/features/norwegian_coral_scaler.joblib
✅ Scaled CORAL-adapted Norwegian features. Preview:
        delta_lat   delta_lon         sog   turn_rate  movement_vector  \
count  515.000000  515.000000  515.000000  515.000000       515.000000   
mean     0.271644    0.876183    0.247032    0.486856         0.199625   
std      0.054037    0.044333    0.333552    0.085856         0.292214   
min      0.000000    0.000000    0.000000    0.000000         0.000000   
25%      0.261972    0.878211    0.025053    0.487630         0.012596   
50%      0.279148    0.883078    0.027580    0.488307         0.015474   
75%      0.283148    0.883606    0.382248    0.488308         0.347333   
max      1.000000    1.000000    1.000000    1.000000         1.000000   

       eez_violation  fishing_zone_dist  
count          5

In [5]:
# Cell 4: Extract 20-Step CORAL-Adapted Norwegian Sequences & Save
print("Cell 4: Extract 20-Step CORAL-Adapted Norwegian Sequences & Save")
print("="*60)

import os
import numpy as np

# The feature columns in df_feat_scaled (now CORAL-adapted)
feature_cols = [
    'delta_lat', 'delta_lon', 'sog',
    'turn_rate', 'movement_vector',
    'eez_violation', 'fishing_zone_dist'
]

SEQ_LEN = 20

# Build list of sequences from CORAL-adapted data
X_seqs = []
vessel_mapping = []  # Keep track of which vessel each sequence belongs to

for mmsi, grp in df_feat_scaled.groupby('mmsi'):
    arr = grp[feature_cols].values
    # Slide a window of length 20
    for i in range(len(arr) - SEQ_LEN + 1):
        X_seqs.append(arr[i : i + SEQ_LEN])
        vessel_mapping.append(mmsi)

# Stack into a single NumPy array
X = np.stack(X_seqs)
vessel_mapping = np.array(vessel_mapping)

print(f"✅ Extracted {X.shape[0]} CORAL-adapted sequences of shape {X.shape[1:]}")
print(f"   Features per timestep: {X.shape[2]}")
print(f"   Vessels represented: {len(np.unique(vessel_mapping))}")

# Save CORAL-adapted sequences to your features folder
features_dir = '/content/drive/MyDrive/C3I-Internship-Work/Programs/MaritimeAnomalyDetection/features'
os.makedirs(features_dir, exist_ok=True)

# Save sequences with CORAL prefix
seq_path = os.path.join(features_dir, 'norwegian_coral_sequences.npy')
np.save(seq_path, X)

# Save vessel mapping
vessel_mapping_path = os.path.join(features_dir, 'norwegian_coral_vessel_mapping.npy')
np.save(vessel_mapping_path, vessel_mapping)

print(f"💾 CORAL-adapted sequences saved to: {seq_path}")
print(f"💾 Vessel mapping saved to: {vessel_mapping_path}")

# Summary of CORAL domain adaptation
print(f"\n🎯 CORAL DOMAIN ADAPTATION SUMMARY:")
print(f"   ✅ Source-Target alignment completed")
print(f"   ✅ {X.shape[0]} sequences ready for model retraining")
print(f"   ✅ Covariance structure aligned for concept drift mitigation")
print(f"   ✅ Files saved with 'coral' prefix for easy identification")
print(f"   📁 Next: Update model training to use CORAL-adapted sequences")

Cell 4: Extract 20-Step CORAL-Adapted Norwegian Sequences & Save
✅ Extracted 361 CORAL-adapted sequences of shape (20, 7)
   Features per timestep: 7
   Vessels represented: 7
💾 CORAL-adapted sequences saved to: /content/drive/MyDrive/C3I-Internship-Work/Programs/MaritimeAnomalyDetection/features/norwegian_coral_sequences.npy
💾 Vessel mapping saved to: /content/drive/MyDrive/C3I-Internship-Work/Programs/MaritimeAnomalyDetection/features/norwegian_coral_vessel_mapping.npy

🎯 CORAL DOMAIN ADAPTATION SUMMARY:
   ✅ Source-Target alignment completed
   ✅ 361 sequences ready for model retraining
   ✅ Covariance structure aligned for concept drift mitigation
   ✅ Files saved with 'coral' prefix for easy identification
   📁 Next: Update model training to use CORAL-adapted sequences
